# Linear Regression

Today you will:
1. Implement linear regression **from scratch** using gradient descent
2. Verify your implementation against the closed-form **Normal Equation**
3. Explore **Ridge**, **Lasso**, and **ElasticNet** regularization with sklearn
4. Evaluate models with **MSE, RMSE, MAE, R²** and diagnose residuals
5. Investigate **polynomial regression**, feature scaling, and multicollinearity


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
import seaborn as sns
import scipy.stats

plt.style.use('seaborn-v0_8')
np.random.seed(42)


## Part 1: Implementing Linear Regression from Scratch

### Step 1: Implement the MSE Loss Function

The mean squared error (MSE) loss for linear regression:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \bigl(h_\theta(x^{(i)}) - y^{(i)}\bigr)^2$$

where $h_\theta(x) = \theta^\top x$ (the bias is absorbed into $\theta$ via a prepended
column of ones in $X$).


In [ ]:
def mse_loss(theta, X, y):
    """
    Compute the MSE loss for linear regression.

    Args:
        theta : Parameter vector            (n_features,)
        X     : Design matrix w/ bias col   (m x n_features)
        y     : Target values               (m,)

    Returns:
        Scalar loss value
    """
    m = len(y)

    # TODO: Implement MSE loss  (~3 lines)
    # 1. Compute predictions  h = X @ theta
    # 2. Compute errors       errors = h - y
    # 3. Return (1 / (2 * m)) * np.dot(errors, errors)
    h      = None
    errors = None
    loss   = None
    return loss


# ── Sanity check ──────────────────────────────────────────────────────────
np.random.seed(0)
_X_chk = np.c_[np.ones(5), np.random.randn(5, 2)]   # 5 x 3
_y_chk = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
_t_chk = np.zeros(3)
_loss  = mse_loss(_t_chk, _X_chk, _y_chk)
_expected = np.mean(_y_chk**2) / 2
print(f'Test loss (theta=0): {_loss:.4f}  '  # should equal expected below
      f'(expected ≈ {_expected:.4f})')


### Step 2: Implement the MSE Gradient

The gradient of $J(\theta)$ with respect to $\theta$ is:

$$\nabla_\theta J = \frac{1}{m} X^\top (X\theta - y)$$

Gradient descent subtracts a fraction $\alpha$ of this vector at each step,
moving $\theta$ toward the minimum.


In [ ]:
def mse_gradient(theta, X, y):
    """
    Compute the gradient of the MSE loss.

    Args:
        theta : Parameter vector            (n_features,)
        X     : Design matrix w/ bias col   (m x n_features)
        y     : Target values               (m,)

    Returns:
        Gradient vector  (n_features,)
    """
    m = len(y)

    # TODO: Implement gradient computation  (~3 lines)
    # 1. Compute predictions  h = X @ theta
    # 2. Compute errors       errors = h - y
    # 3. Return (1 / m) * X.T @ errors
    h        = None
    errors   = None
    gradient = None
    return gradient


# ── Sanity check ──────────────────────────────────────────────────────────
_grad = mse_gradient(_t_chk, _X_chk, _y_chk)
print(f'Gradient at theta=0: {_grad}')
print(f'Shape: {_grad.shape}  (should be (3,))')


### Step 3: Create a Dataset

We generate a 1-feature synthetic dataset for easy visualization, then prepend
a column of ones to $X$ so the first element of $\theta$ acts as the bias/intercept.


In [ ]:
# Generate a 1-feature regression dataset
X_raw, y = make_regression(n_samples=300, n_features=1, noise=20, random_state=42)

# Prepend bias column (column of ones)
X = np.c_[np.ones(X_raw.shape[0]), X_raw]

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train_raw = X_train[:, 1:]   # raw features, for sklearn & plots
X_test_raw  = X_test[:, 1:]

print(f'Training set : {X_train.shape[0]} samples, '
      f'{X_train.shape[1]} features (incl. bias)')
print(f'Test set     : {X_test.shape[0]} samples')

# Scatter plot
plt.figure(figsize=(7, 5))
plt.scatter(X_train_raw, y_train, alpha=0.5, label='Train', color='steelblue')
plt.scatter(X_test_raw,  y_test,  alpha=0.5, label='Test',  color='tomato')
plt.xlabel('Feature 1')
plt.ylabel('Target')
plt.title('Synthetic Regression Dataset')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### Step 4: Implement Gradient Descent

We minimize $J(\theta)$ by repeatedly applying the update rule:

$$\theta := \theta - \alpha \,\nabla_\theta J(\theta)$$

where $\alpha$ is the **learning rate**.  Each step moves $\theta$ a small distance in the direction of steepest descent.


In [ ]:
class LinearRegressionFromScratch:
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        self.learning_rate  = learning_rate
        self.max_iterations = max_iterations
        self.tolerance      = tolerance
        self.theta          = None
        self.cost_history   = []

    def fit(self, X, y):
        n_features      = X.shape[1]
        self.theta      = np.random.normal(0, 0.01, n_features)
        self.cost_history = []

        for i in range(self.max_iterations):
            # TODO: Fill in the 4 lines below  (~4 lines)
            # 1. Compute current loss :  loss = mse_loss(self.theta, X, y)
            # 2. Append to history    :  self.cost_history.append(loss)
            # 3. Compute gradient     :  gradient = mse_gradient(self.theta, X, y)
            # 4. Update theta         :  self.theta -= self.learning_rate * gradient
            loss     = None
            gradient = None

            # ── early stopping (leave this block as-is) ──────────────────
            if i > 0 and abs(self.cost_history[-2] - self.cost_history[-1]) < self.tolerance:
                print(f'Converged at iteration {i}')
                break

        return self

    def predict(self, X):
        return X @ self.theta

    def score(self, X, y):
        y_pred  = self.predict(X)
        ss_res  = np.sum((y - y_pred) ** 2)
        ss_tot  = np.sum((y - np.mean(y)) ** 2)
        return 1 - ss_res / ss_tot


# ── Train ─────────────────────────────────────────────────────────────────
model_scratch = LinearRegressionFromScratch(learning_rate=0.01, max_iterations=2000)
model_scratch.fit(X_train, y_train)

print(f'\nFinal loss : {model_scratch.cost_history[-1]:.4f}')
print(f'Train R²   : {model_scratch.score(X_train, y_train):.4f}')
print(f'Test  R²   : {model_scratch.score(X_test,  y_test):.4f}')
print(f'Theta (bias, w): {model_scratch.theta}')


### Step 5: Visualize Training Progress and Fitted Line


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: cost history
axes[0].plot(model_scratch.cost_history, color='steelblue')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Gradient Descent: Cost History')
axes[0].grid(True)

# Right: fitted line
x_line = np.linspace(X_train_raw.min(), X_train_raw.max(), 100)
X_line = np.c_[np.ones(100), x_line]
y_line = model_scratch.predict(X_line)

axes[1].scatter(X_train_raw, y_train, alpha=0.5, color='steelblue', label='Train data')
axes[1].plot(x_line, y_line, color='tomato', linewidth=2, label='Fitted line')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Target')
axes[1].set_title('Fitted Regression Line')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


### Bonus: Normal Equation (Closed-Form Ground-Truth Checker)

Setting $\nabla_\theta J = 0$ and solving analytically gives:

$$\theta^* = (X^\top X)^{-1} X^\top y$$

This is the **exact** optimum in a single step (no learning rate, no iterations).
We use it to verify that your gradient descent implementation converged to the right answer.


In [ ]:
# Normal Equation — closed-form solution
theta_normal = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train

# Compare
print('=' * 52)
print(f'{"":20s}  {"Bias":>10s}  {"Coef":>10s}')
print('-' * 52)
print(f'{"Normal Equation":20s}  '
      f'{theta_normal[0]:10.4f}  {theta_normal[1]:10.4f}')
print(f'{"Gradient Descent":20s}  '
      f'{model_scratch.theta[0]:10.4f}  {model_scratch.theta[1]:10.4f}')
print(f'{"Difference":20s}  '
      f'{abs(theta_normal[0]-model_scratch.theta[0]):10.6f}  '
      f'{abs(theta_normal[1]-model_scratch.theta[1]):10.6f}')
print('=' * 52)

y_pred_normal = X_test @ theta_normal
print(f'\nR\u00b2 (Normal Equation) : {r2_score(y_test, y_pred_normal):.4f}')
print(f'R\u00b2 (Gradient Descent): {model_scratch.score(X_test, y_test):.4f}')
print('\nIf your GD implementation is correct, '
      'the theta values should be very close!')


## Part 2: Exploring sklearn's Linear Models

Now let's compare our from-scratch implementation with sklearn and explore
Ridge, Lasso, and ElasticNet regularization.


In [ ]:
# sklearn's LinearRegression (uses the Normal Equation internally)
model_sklearn = LinearRegression()
model_sklearn.fit(X_train_raw, y_train)

y_pred_sklearn = model_sklearn.predict(X_test_raw)
r2_sklearn     = r2_score(y_test, y_pred_sklearn)

print('Comparison: From Scratch vs sklearn')
print('=' * 57)
print(f'{"":20s}  {"Intercept":>10s}  {"Coef":>10s}  {"R\u00b2":>6s}')
print('-' * 57)
print(f'{"From Scratch (GD)":20s}  '
      f'{model_scratch.theta[0]:10.4f}  '
      f'{model_scratch.theta[1]:10.4f}  '
      f'{model_scratch.score(X_test, y_test):6.4f}')
print(f'{"sklearn OLS":20s}  '
      f'{model_sklearn.intercept_:10.4f}  '
      f'{model_sklearn.coef_[0]:10.4f}  '
      f'{r2_sklearn:6.4f}')
print('=' * 57)


### Ridge Regression (L2 Regularization)

Ridge adds a penalty on the squared magnitude of the weights:

$$J_{\text{Ridge}}(\theta) = \frac{1}{2m}\|X\theta - y\|^2 + \alpha\|\theta\|_2^2$$

Larger $\alpha$ → stronger regularization → coefficients shrink toward zero.
This trades a little bias for lower variance, which can improve generalisation.


In [ ]:
alphas         = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_r2_train = []
ridge_r2_test  = []
ridge_coef_norms = []

for alpha in alphas:
    # TODO: Fill in the loop body  (~5 lines)
    # 1. Create  Ridge(alpha=alpha)
    # 2. Fit on  (X_train_raw, y_train)
    # 3. Append train R² to ridge_r2_train
    # 4. Append test  R² to ridge_r2_test
    # 5. Append np.linalg.norm(model.coef_) to ridge_coef_norms
    pass


# ── Plots (pre-filled) ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogx(alphas, ridge_r2_train, 'o-', label='Train R\u00b2')
axes[0].semilogx(alphas, ridge_r2_test,  's--', label='Test R\u00b2')
axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('R\u00b2')
axes[0].set_title('Ridge: R\u00b2 vs Alpha')
axes[0].legend()
axes[0].grid(True)

axes[1].semilogx(alphas, ridge_coef_norms, 'o-', color='tomato')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('\u2016w\u2016₂')
axes[1].set_title('Ridge: Coefficient Norm vs Alpha')
axes[1].grid(True)

plt.tight_layout()
plt.show()


### Lasso Regression (L1 Regularization)

Lasso adds a penalty on the **absolute** magnitude of the weights:

$$J_{\text{Lasso}}(\theta) = \frac{1}{2m}\|X\theta - y\|^2 + \alpha\|\theta\|_1$$

The L1 penalty induces **sparsity**: many coefficients are driven exactly to zero,
making Lasso useful for automatic **feature selection**.


In [ ]:
# Switch to a 10-feature dataset so sparsity effects are visible
X10_raw, y10 = make_regression(
    n_samples=300, n_features=10, n_informative=5, noise=20, random_state=42)
X10_train_raw, X10_test_raw, y10_train, y10_test = train_test_split(
    X10_raw, y10, test_size=0.2, random_state=42)

lasso_alphas  = [0.01, 0.1, 1, 10, 50, 100]
lasso_r2_test = []
lasso_coefs   = []

for alpha in lasso_alphas:
    # TODO: Fill in the loop body  (~5 lines)
    # 1. Create  Lasso(alpha=alpha, max_iter=5000)
    # 2. Fit on  (X10_train_raw, y10_train)
    # 3. Compute test R² with r2_score(y10_test, model.predict(X10_test_raw))
    # 4. Append test R² to lasso_r2_test
    # 5. Append model.coef_  (length-10 array) to lasso_coefs
    pass

lasso_coefs = np.array(lasso_coefs)   # shape: (n_alphas, n_features)


# ── Plots (pre-filled) ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    lasso_coefs, ax=axes[0],
    xticklabels=[f'f{i}' for i in range(10)],
    yticklabels=[str(a) for a in lasso_alphas],
    cmap='RdBu_r', center=0)
axes[0].set_xlabel('Feature')
axes[0].set_ylabel('Alpha')
axes[0].set_title('Lasso Coefficients (rows = increasing alpha)')

n_nonzero = [int(np.sum(np.abs(row) > 1e-4)) for row in lasso_coefs]
axes[1].plot(lasso_alphas, n_nonzero, 'o-', color='steelblue')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('# Non-zero Coefficients')
axes[1].set_title('Lasso: Sparsity vs Alpha')
axes[1].grid(True)

plt.tight_layout()
plt.show()


### ElasticNet (Blended L1 + L2 Regularization)

ElasticNet combines both penalties:

$$J_{\text{EN}}(\theta) = \frac{1}{2m}\|X\theta - y\|^2
  + \alpha\!\left[\rho\|\theta\|_1 + \frac{1-\rho}{2}\|\theta\|_2^2\right]$$

where $\rho$ (`l1_ratio` in sklearn) controls the L1/L2 mix.
$\rho=1$ → pure Lasso; $\rho=0$ → pure Ridge.


In [ ]:
models_cmp = {
    'LinearRegression'      : LinearRegression(),
    'Ridge(alpha=1)'        : Ridge(alpha=1.0),
    'Lasso(alpha=1)'        : Lasso(alpha=1.0, max_iter=5000),
    'ElasticNet(a=1,rho=.5)': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000),
}

print(f'{"Model":30s}  {"R\u00b2":>6s}  {"\u2016w\u2016\u2082":>8s}  {"# Zeros":>8s}')
print('-' * 60)
for name, m in models_cmp.items():
    m.fit(X10_train_raw, y10_train)
    r2   = r2_score(y10_test, m.predict(X10_test_raw))
    norm = np.linalg.norm(m.coef_)
    nz   = int(np.sum(np.abs(m.coef_) < 1e-4))
    print(f'{name:30s}  {r2:6.4f}  {norm:8.4f}  {nz:8d}')


## Part 3: Evaluation and Residual Analysis

A good regression model should have:
- Low MSE / RMSE / MAE on the held-out test set
- Residuals that are **normally distributed** and **randomly scattered** around zero
  (no systematic pattern)


In [ ]:
y_pred_test = model_scratch.predict(X_test)

# TODO: Compute the four metrics below  (~4 lines)
# Replace each None with the correct expression.
# Hint: use mean_squared_error, mean_absolute_error, r2_score (already imported)
#       RMSE = sqrt(MSE)

mse  = None   # mean_squared_error(y_test, y_pred_test)
rmse = None   # np.sqrt(mse)
mae  = None   # mean_absolute_error(y_test, y_pred_test)
r2   = None   # r2_score(y_test, y_pred_test)

print('Test-set Metrics  (from-scratch GD model)')
print('=' * 42)
print(f'  MSE  : {mse:.4f}')
print(f'  RMSE : {rmse:.4f}')
print(f'  MAE  : {mae:.4f}')
print(f'  R\u00b2   : {r2:.4f}')


### Residual Analysis

Residuals are $e^{(i)} = y^{(i)} - \hat{y}^{(i)}$.  A well-fitted linear model should show:

1. **Residuals vs. Fitted values**: random scatter around 0 (no funnel/curve pattern)
2. **Histogram**: roughly bell-shaped, centred at 0
3. **Q-Q plot**: residual quantiles fall near the diagonal (normality)


In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Residuals vs Fitted
axes[0].scatter(y_pred_test, residuals, alpha=0.6, color='steelblue', s=25)
axes[0].axhline(0, color='tomato', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
axes[0].grid(True)

# Panel 2: Histogram
axes[1].hist(residuals, bins=25, edgecolor='black', color='steelblue', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Histogram of Residuals')
axes[1].grid(True)

# Panel 3: Q-Q plot
scipy.stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals')
axes[2].grid(True)

plt.suptitle('Residual Diagnostics — From-Scratch GD Model', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Fit Ridge and Lasso on the 1-feature dataset for a side-by-side comparison
model_ridge_cmp = Ridge(alpha=1.0)
model_lasso_cmp = Lasso(alpha=1.0, max_iter=5000)
model_ridge_cmp.fit(X_train_raw, y_train)
model_lasso_cmp.fit(X_train_raw, y_train)

models_for_resid = {
    'GD Scratch' : (y_test, model_scratch.predict(X_test)),
    'sklearn OLS': (y_test, model_sklearn.predict(X_test_raw)),
    'Ridge(a=1)' : (y_test, model_ridge_cmp.predict(X_test_raw)),
    'Lasso(a=1)' : (y_test, model_lasso_cmp.predict(X_test_raw)),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (yt, yp)) in zip(axes, models_for_resid.items()):
    res = yt - yp
    ax.scatter(yp, res, alpha=0.5, color='steelblue', s=20)
    ax.axhline(0, color='tomato', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Fitted')
    ax.set_ylabel('Residual')
    ax.set_title(f'{name}\nR\u00b2={r2_score(yt, yp):.3f}')
    ax.grid(True)

plt.suptitle('Residuals vs Fitted — Model Comparison', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## Part 4: Open-Ended Exploration

### Exploration A: Polynomial Regression and the Bias-Variance Trade-off

By expanding features with `PolynomialFeatures`, a linear model can fit non-linear data.
Higher-degree polynomials are more flexible but risk **overfitting** (high variance).

The true data-generating process below is **quadratic**, so degree 2 should fit best.


In [ ]:
# True DGP is quadratic: y = 1.5*x^2 - 2*x + noise
np.random.seed(42)
n_poly   = 80
X_poly_raw = np.linspace(-3, 3, n_poly).reshape(-1, 1)
y_poly     = (1.5 * X_poly_raw.ravel()**2
              - 2  * X_poly_raw.ravel()
              + np.random.randn(n_poly) * 3)

X_poly_train, X_poly_test, y_poly_train, y_poly_test = train_test_split(
    X_poly_raw, y_poly, test_size=0.25, random_state=0)

degrees    = [1, 2, 3, 5, 9, 15]
train_r2s  = []
test_r2s   = []

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
x_plot = np.linspace(-3.2, 3.2, 200).reshape(-1, 1)

for i, degree in enumerate(degrees):
    # TODO: Fill in the loop body  (~10 lines)
    # 1.  poly = PolynomialFeatures(degree=degree, include_bias=False)
    # 2.  Transform training and test data (fit_transform / transform)
    # 3.  Fit  LinearRegression()  on the transformed training data
    # 4.  Compute and record train_r2  →  train_r2s
    # 5.  Compute and record test_r2   →  test_r2s
    # 6.  Transform x_plot and predict  y_plot
    # 7.  In axes[i]:
    #       scatter X_poly_train vs y_poly_train  (color='steelblue', alpha=0.5)
    #       scatter X_poly_test  vs y_poly_test   (color='tomato',    alpha=0.5)
    #       plot    x_plot       vs y_plot         (color='black', lw=2)
    #       set_title(f'Degree {degree}  |  Test R\u00b2={test_r2:.3f}')
    #       axes[i].set_ylim(-10, 30)    # keep y-axis stable
    pass

plt.suptitle('Polynomial Regression: Bias-Variance Trade-off', fontsize=13)
plt.tight_layout()
plt.show()

# Summary
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.plot(degrees, train_r2s, 'o-', label='Train R\u00b2')
ax2.plot(degrees, test_r2s,  's--', label='Test R\u00b2')
ax2.set_xlabel('Polynomial Degree')
ax2.set_ylabel('R\u00b2')
ax2.set_title('Bias-Variance: Train vs Test R\u00b2')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


### Exploration B: Feature Scaling and Gradient Descent Convergence

Features on different scales cause the loss surface to be **elongated**, making
gradient descent zig-zag and converge slowly (or require a tiny learning rate).
Standardising features to zero mean and unit variance corrects this.


In [ ]:
# Two-scale dataset: feature 0 in [0,1], feature 1 in [0,1000]
np.random.seed(7)
n_scale = 200
X_s1    = np.random.rand(n_scale)            # scale  ~1
X_s2    = np.random.rand(n_scale) * 1000     # scale  ~1000
y_scale = 3 * X_s1 + 0.002 * X_s2 + np.random.randn(n_scale) * 0.5

X_unscaled = np.c_[X_s1, X_s2]
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X_unscaled)

X_us_b = np.c_[np.ones(n_scale), X_unscaled]
X_sc_b = np.c_[np.ones(n_scale), X_scaled]

model_unscaled = LinearRegressionFromScratch(learning_rate=1e-7, max_iterations=500)
model_scaled   = LinearRegressionFromScratch(learning_rate=0.1,  max_iterations=500)
model_unscaled.fit(X_us_b, y_scale)
model_scaled.fit(X_sc_b,   y_scale)

plt.figure(figsize=(9, 4))
plt.plot(model_unscaled.cost_history, label='Unscaled (lr=1e-7)', color='tomato')
plt.plot(model_scaled.cost_history,   label='Scaled   (lr=0.1)',  color='steelblue')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Convergence: Scaled vs Unscaled Features')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f'Final loss (unscaled): {model_unscaled.cost_history[-1]:.6f}')
print(f'Final loss (scaled)  : {model_scaled.cost_history[-1]:.6f}')


### Exploration C: Multicollinearity

When features are highly correlated, $(X^\top X)$ becomes nearly singular and OLS
coefficients become **unstable** (large, noisy, sensitive to small data changes).
Ridge regression stabilises them by adding $\alpha I$ before inversion:

$$\theta^* = (X^\top X + \alpha I)^{-1} X^\top y$$


In [ ]:
np.random.seed(0)
n_mc = 200
x_mc = np.random.randn(n_mc)
X_mc = np.column_stack([
    x_mc,
    x_mc + 0.01 * np.random.randn(n_mc),   # nearly identical to x_mc
    np.random.randn(n_mc),                   # independent feature
])
y_mc = 2 * X_mc[:, 0] + 3 * X_mc[:, 2] + np.random.randn(n_mc)

X_mc_train, X_mc_test, y_mc_train, y_mc_test = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=0)

alphas_mc = [0, 0.01, 0.1, 1, 10, 100]
print(f'{"Alpha":>8s}  {"coef_0":>10s}  {"coef_1":>10s}  {"coef_2":>10s}  {"Test R\u00b2":>8s}')
print('-' * 56)
for a in alphas_mc:
    m = LinearRegression() if a == 0 else Ridge(alpha=a)
    m.fit(X_mc_train, y_mc_train)
    r2_mc = r2_score(y_mc_test, m.predict(X_mc_test))
    c = m.coef_
    print(f'{a:8.2f}  {c[0]:10.3f}  {c[1]:10.3f}  {c[2]:10.3f}  {r2_mc:8.4f}')

print('\nNotice: as alpha increases, '
      'the two correlated coefficients (coef_0, coef_1) stabilise.')


## Further Questions

1. **Learning rate sensitivity** — What happens when you set `learning_rate=1.0`? What about `1e-10`? Explain the behaviour in terms of the shape of the loss surface.

2. **Regularisation from scratch** — Add an L2 penalty term to `mse_loss` and `mse_gradient` and re-train `LinearRegressionFromScratch`. Compare the resulting coefficients with sklearn's `Ridge`.

3. **Stochastic Gradient Descent** — Modify `LinearRegressionFromScratch.fit` to process one random sample at a time (SGD) instead of the full batch. Does it converge faster or slower in terms of wall-clock iterations?

4. **UCI Diabetes Dataset** — Load `sklearn.datasets.load_diabetes()` and fit all four models (OLS, Ridge, Lasso, ElasticNet). Which features does Lasso zero out? Do those features make clinical sense?

5. **Polynomial + Ridge** — High-degree polynomial regression overfits. Combine `PolynomialFeatures(degree=9)` with `Ridge` and sweep over `alpha` values. At what point does regularisation rescue the model from overfitting?

6. **Residuals revisited** — Generate a dataset where the true relationship is non-linear (e.g., $y = \sin(x) + \epsilon$) and fit a linear model. What do the residual plots reveal about the model misspecification?
